In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import rasterio

In [ ]:
!ln -s "/content/drive/My Drive/OpenEarthMap_wo_xBD" "/content/LC"

In [ ]:
# class for images, uses paths present in _paths lists to extract path values
class img:
  def __init__(self, path):
    self.path = path
    self.raw = "/content/LC/" + path.split('_')[0] + "/images/" + path
    self.label = "/content/LC/" + path.split('_')[0] + "/labels/" + path

# load images as tensors
def read_tif_to_numpy(path):
    with rasterio.open(path) as src:
        arr = src.read()  # (C, H, W)

    # Convert to (H, W, C)
    arr = np.transpose(arr, (1, 2, 0))
    return arr

In [ ]:
# load models
from keras.models import load_model
Unet = load_model('/content/drive/MyDrive/Data-5610-Advanced-Machine-Learning/U_Net_Sat_Segementation.h5')
Deeplab = load_model('/content/drive/MyDrive/Data 5610 Advanced Machine Learning/Projects/Final_Project/DeepLabv3+_Sat_Segementation.h5')

# Comparison between labels masks

In [ ]:
# Create prediction masks for select images
def compare_mask(img_path, size = (512,512)):
  # convert raw img to a tensor
  raw_img_arr = read_tif_to_numpy(img(img_path).raw)

  # resize img with lanzcos3
  img_arr = tf.image.resize(
                raw_img_arr,
                (size[0], size[1]), # Pass only height and width
                method=tf.image.ResizeMethod.LANCZOS3
            ).numpy().astype(np.float32)

  # Normalize input to [0, 1] range to match training data
  img_arr = img_arr / 255.0

  # Unet Predict
  Upred = Unet.predict(img_arr[None, ...])[0]
  Upred_mask_arr = np.argmax(Upred, axis=-1)

  # DeepLab Predict
  Dpred = Deeplab.predict(img_arr[None, ...])[0]
  Dpred_mask_arr = np.argmax(Dpred, axis=-1)

  # plot
  plt.figure(figsize=(16, 5))

  plt.subplot(1, 4, 1)
  plt.title(img_path[:-4])
  plt.imshow(tf.keras.utils.array_to_img(img_arr))
  plt.axis('off')

  plt.subplot(1, 4, 2)
  plt.title("Ground Truth Label Mask")
  plt.imshow(read_tif_to_numpy(img(img_path).label))
  plt.axis("off")

  plt.subplot(1, 4, 3)
  plt.title("Unet Predicted Label Mask")
  plt.imshow(Upred_mask_arr)
  plt.axis("off")

  plt.subplot(1, 4, 4)
  plt.title("DeepLabv3+ Predicted Label Mask")
  plt.imshow(Dpred_mask_arr)
  plt.axis("off")
  plt.show()

In [ ]:
comp_imgs = ["paris_67",
             'aachen_16',
             "chicago_10",
             'kyoto_42',
             'vienna_16'
             ]
for image_name in comp_imgs:
  compare_mask(image_name + '.tif')

#Predictions on Unseen (Unlabeled Images)

In [ ]:
# Create prediction masks for select images
def pred_mask(img_path, size = (512,512)):
  # convert raw img to a tensor
  raw_img_arr = read_tif_to_numpy(img(img_path).raw)

  # resize img with lanzcos3
  img_arr = tf.image.resize(
                raw_img_arr,
                (size[0], size[1]), # Pass only height and width
                method=tf.image.ResizeMethod.LANCZOS3
            ).numpy().astype(np.float32)

  # Normalize input to [0, 1] range to match training data
  img_arr = img_arr / 255.0

  # Unet Predict
  Upred = Unet.predict(img_arr[None, ...])[0]
  Upred_mask_arr = np.argmax(Upred, axis=-1)

  # DeepLab Predict
  Dpred = Deeplab.predict(img_arr[None, ...])[0]
  Dpred_mask_arr = np.argmax(Dpred, axis=-1)

  # plot
  plt.figure(figsize=(12, 5))

  plt.subplot(1, 3, 1)
  plt.title(img_path[:-4])
  plt.imshow(tf.keras.utils.array_to_img(img_arr))
  plt.axis('off')

  plt.subplot(1, 3, 2)
  plt.title("Unet Predicted Label Mask")
  plt.imshow(Upred_mask_arr)
  plt.axis("off")

  plt.subplot(1, 3, 3)
  plt.title("DeepLabv3+ Predicted Label Mask")
  plt.imshow(Dpred_mask_arr)
  plt.axis("off")
  plt.show()

Interesting Observations:
* dortmund_2
* mahe_15: Ship gets classified as a building
* monrovia_21/12
* koeln_12: Buildings obfuscated
* paris is just interesting general, it was resized from 650 not 1024, and had interesting angles/lighting
* santiago_59
* daressalaam_50
* koeln_29/26
* baybay_1
* zanzibar_45
* austin_47: big texas road, hetrogenous surroundings
* tyrolw_67: interesting angle
* accra_10: small houses

In [ ]:
# print out specific images for comparison
img_lst = ["austin_47", #big texas road, hetrogenous surroundings
           "dortmund_2",
           "mahe_15", #Ship gets classified as a building
           "monrovia_21",
           "koeln_12", #Buildings obfuscated
           "paris_67", #Paris was resized from 650 not 1024, and has interesting angles/lighting
           "santiago_59",
           "daressalaam_50",
           "koeln_29",
           "baybay_1",
           "zanzibar_45",
           "tyrolw_67", #interesting angle
           "accra_10" #small houses
           ]

for image_name in img_lst:
  pred_mask(image_name + '.tif')